In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

# Day 05.1 · Balanced Tree Baselines

Notebook único de evidencia para la iteración corta que aísla la hipótesis de balanceo nativo en árboles.

Objetivos:
- comparar `LIGHTGBM` y `XGBOOST` balanceados nativamente contra el baseline corregido `LR_smote_0.5`;
- comparar cada pareja contra su versión Day 05 no balanceada;
- decidir si aparece alguna pareja `defendible` o `casi-gate` antes de Brent.

## Contexto y contrato fijo

- `cutoff_date`: `2028-02-21`
- holdout oficial: mismo universo post-Day 04 rerun
- métricas oficiales: `accuracy`, `balanced_accuracy`, `f1_pos`, `top1_hit`, `top2_hit`, `coverage`
- scope corto: sin ETL nuevo, sin nuevas features, sin `ROS`, sin `SMOTE`, sin Brent
- el resultado se interpreta como complemento metodológico de Day 05, no como reescritura de `notebook 16`

In [2]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "artifacts").exists() and (PROJECT_ROOT.parent / "artifacts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DAY05_ROOT = PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05"
DAY051_ROOT = PROJECT_ROOT / "artifacts" / "public" / "metrics" / "day05_1"
DAY05_SUMMARIES = sorted(DAY05_ROOT.glob("*_run_summary.json"), key=lambda path: path.stat().st_mtime)
DAY051_SUMMARIES = sorted(DAY051_ROOT.glob("*_run_summary.json"), key=lambda path: path.stat().st_mtime)
if not DAY05_SUMMARIES:
    raise FileNotFoundError("No hay run summaries Day 05 todavía.")
if not DAY051_SUMMARIES:
    raise FileNotFoundError("No hay run summaries Day 05.1 todavía. Ejecuta el runner público antes de usar el notebook.")

DAY05_RUN_ID = json.loads(DAY05_SUMMARIES[-1].read_text(encoding="utf-8"))["run_id"]
DAY051_RUN_ID = json.loads(DAY051_SUMMARIES[-1].read_text(encoding="utf-8"))["run_id"]
day05_df = pd.read_csv(DAY05_ROOT / f"{DAY05_RUN_ID}_canonical_candidates.csv")
day051_df = pd.read_csv(DAY051_ROOT / f"{DAY051_RUN_ID}_canonical_candidates.csv")
selection_payload = json.loads((DAY051_ROOT / f"{DAY051_RUN_ID}_selection_decisions.json").read_text(encoding="utf-8"))
policy_payload = json.loads((DAY051_ROOT / f"{DAY051_RUN_ID}_policy_summary.json").read_text(encoding="utf-8"))
DAY05_RUN_ID, DAY051_RUN_ID, day05_df.shape, day051_df.shape

('20260306T_day05_run01', '20260306T_day05_run02', (15, 35), (6, 42))

## Evidencia principal vs baseline corregido

La lectura principal sigue siendo siempre `modelo puro vs baseline corregido`.

In [3]:
main_cols = [
    'dataset_alias', 'model_family', 'model_variant', 'top2_hit', 'balanced_accuracy', 'coverage',
    'delta_top2_vs_baseline', 'delta_bal_acc_vs_baseline', 'defendible', 'almost_gate', 'followup_decision'
]
day051_df[main_cols].sort_values(
    ['defendible', 'almost_gate', 'delta_bal_acc_vs_baseline', 'delta_top2_vs_baseline'],
    ascending=[False, False, False, False],
).reset_index(drop=True)

,dataset_alias,model_family,model_variant,top2_hit,balanced_accuracy,coverage,delta_top2_vs_baseline,delta_bal_acc_vs_baseline,defendible,almost_gate,followup_decision
0,V2_TRANSPORT_ONLY,LIGHTGBM,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,0.882643,0.881431,1.0,0.024307,0.016347,True,False,promoted_pure_model_open_short_tuning
1,V2,LIGHTGBM,V2_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1,0.889100,0.874296,1.0,0.030764,0.009212,False,True,consider_future_tuning_or_resampling
2,V2_TRANSPORT_CARRY30D_ONLY,LIGHTGBM,V2_TRANSPORT_CARRY30D_ONLY_LIGHTGBM_CLASS_WEIG...,0.890999,0.873831,1.0,0.032663,0.008747,False,True,consider_future_tuning_or_resampling
3,V2,XGBOOST,V2_XGBOOST_SCALE_POS_WEIGHT_v1,0.876187,0.869778,1.0,0.017851,0.004694,False,True,consider_future_tuning_or_resampling
4,V2_TRANSPORT_CARRY30D_ONLY,XGBOOST,V2_TRANSPORT_CARRY30D_ONLY_XGBOOST_SCALE_POS_W...,0.869351,0.867433,1.0,0.011015,0.002349,False,False,close_tabular
5,V2_TRANSPORT_ONLY,XGBOOST,V2_TRANSPORT_ONLY_XGBOOST_SCALE_POS_WEIGHT_v1,0.868971,0.863573,1.0,0.010635,-0.001511,False,False,close_tabular


## Comparación directa contra Day 05 no balanceado

Esta tabla responde a la hipótesis exacta de Day 05.1: si el balanceo nativo desde el arranque corrige parte del deterioro visto en Day 05.

In [4]:
compare_cols = [
    'dataset_alias', 'model_family', 'day05_unbalanced_variant', 'model_variant',
    'delta_top2_vs_day05_unbalanced', 'delta_bal_acc_vs_day05_unbalanced', 'delta_coverage_vs_day05_unbalanced',
    'top2_hit', 'balanced_accuracy', 'defendible', 'almost_gate'
]
day051_df[compare_cols].sort_values(
    ['delta_bal_acc_vs_day05_unbalanced', 'delta_top2_vs_day05_unbalanced'],
    ascending=False,
).reset_index(drop=True)

,dataset_alias,model_family,day05_unbalanced_variant,model_variant,delta_top2_vs_day05_unbalanced,delta_bal_acc_vs_day05_unbalanced,delta_coverage_vs_day05_unbalanced,top2_hit,balanced_accuracy,defendible,almost_gate
0,V2_TRANSPORT_ONLY,LIGHTGBM,V2_TRANSPORT_ONLY_LIGHTGBM_v1,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,0.001899,0.141861,0.0,0.882643,0.881431,True,False
1,V2,LIGHTGBM,V2_LIGHTGBM_v1,V2_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1,0.017471,0.130750,0.0,0.889100,0.874296,False,True
2,V2_TRANSPORT_ONLY,XGBOOST,V2_TRANSPORT_ONLY_XGBOOST_v1,V2_TRANSPORT_ONLY_XGBOOST_SCALE_POS_WEIGHT_v1,-0.013293,0.125756,0.0,0.868971,0.863573,False,False
3,V2_TRANSPORT_CARRY30D_ONLY,LIGHTGBM,V2_TRANSPORT_CARRY30D_ONLY_LIGHTGBM_v1,V2_TRANSPORT_CARRY30D_ONLY_LIGHTGBM_CLASS_WEIG...,0.007976,0.125387,0.0,0.890999,0.873831,False,True
4,V2_TRANSPORT_CARRY30D_ONLY,XGBOOST,V2_TRANSPORT_CARRY30D_ONLY_XGBOOST_v1,V2_TRANSPORT_CARRY30D_ONLY_XGBOOST_SCALE_POS_W...,-0.015571,0.114685,0.0,0.869351,0.867433,False,False
5,V2,XGBOOST,V2_XGBOOST_v1,V2_XGBOOST_SCALE_POS_WEIGHT_v1,0.001899,0.099094,0.0,0.876187,0.869778,False,True


## Contexto adicional vs LR equivalente

La comparación contra LR equivalente queda como lectura contextual adicional, no como trigger principal de decisión.

In [5]:
secondary_cols = [
    'dataset_alias', 'lr_equivalent_variant', 'model_family', 'model_variant',
    'delta_top2_vs_lr_equivalente', 'delta_bal_acc_vs_lr_equivalente', 'top2_hit', 'balanced_accuracy'
]
day051_df[secondary_cols].sort_values(
    ['delta_bal_acc_vs_lr_equivalente', 'delta_top2_vs_lr_equivalente'],
    ascending=False,
).reset_index(drop=True)

,dataset_alias,lr_equivalent_variant,model_family,model_variant,delta_top2_vs_lr_equivalente,delta_bal_acc_vs_lr_equivalente,top2_hit,balanced_accuracy
0,V2_TRANSPORT_ONLY,V2_TRANSPORT_ONLY_LR_smote_0.5_v1,LIGHTGBM,V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANC...,0.022408,0.015584,0.882643,0.881431
1,V2_TRANSPORT_CARRY30D_ONLY,V2_TRANSPORT_CARRY30D_ONLY_LR_smote_0.5_v1,LIGHTGBM,V2_TRANSPORT_CARRY30D_ONLY_LIGHTGBM_CLASS_WEIG...,0.028485,0.009750,0.890999,0.873831
2,V2,LR_smote_0.5,LIGHTGBM,V2_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1,0.030764,0.009212,0.889100,0.874296
3,V2,LR_smote_0.5,XGBOOST,V2_XGBOOST_SCALE_POS_WEIGHT_v1,0.017851,0.004694,0.876187,0.869778
4,V2_TRANSPORT_CARRY30D_ONLY,V2_TRANSPORT_CARRY30D_ONLY_LR_smote_0.5_v1,XGBOOST,V2_TRANSPORT_CARRY30D_ONLY_XGBOOST_SCALE_POS_W...,0.006837,0.003352,0.869351,0.867433
5,V2_TRANSPORT_ONLY,V2_TRANSPORT_ONLY_LR_smote_0.5_v1,XGBOOST,V2_TRANSPORT_ONLY_XGBOOST_SCALE_POS_WEIGHT_v1,0.008736,-0.002274,0.868971,0.863573


In [6]:
selection_payload

{'run_id': '20260306T_day05_run02',
 'defendible_variants': ['V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1'],
 'almost_gate_variants': ['V2_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1',
  'V2_TRANSPORT_CARRY30D_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1',
  'V2_XGBOOST_SCALE_POS_WEIGHT_v1'],
 'best_balanced_variant': 'V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1',
 'pre_policy_close_decision': 'open_single_policy_check_then_decide_tuning',
 'close_decision': 'promote_pure_model_open_short_tuning',
 'final_close_decision': 'promote_pure_model_open_short_tuning',
 'pure_model_decision': 'promote_model_champion',
 'current_model_champion_variant': 'V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1',
 'policy_decision': 'reject_new_policy_keep_previous_operational_policy',
 'current_operational_policy_variant': 'V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1',
 'rejected_policy_variant': 'V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALA

In [7]:
policy_payload

{'run_id': '20260306T_day05_run02',
 'executed': True,
 'best_final_pure_variant': 'V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1',
 'policy_variant': 'V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1',
 'policy_metrics_path': './artifacts/public/metrics/candidates/20260306/20260306T_day05_run02_V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1_metrics.json',
 'policy_summary_path': './artifacts/public/postinferencia/audits/20260306/20260306T_day05_run02_V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1_policy_input_summary_assist_PRODUCT_002_follow_PRODUCT_003_SUPPLIER_009_20260306T_day05_run02_policy.json',
 'promotion_decision': 'keep_baseline',
 'policy_decision': 'reject_new_policy_keep_previous_operational_policy',
 'current_operational_policy_variant': 'V2_TRANSPORT_ONLY_LR_smote_0.5_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLL

## Decisión de cierre Day 05.1

Checklist final a dejar respondido tras la ejecución:
- si aparece o no algún candidato `defendible`;
- si aparece o no algún candidato `casi-gate`;
- si la hipótesis de balanceo nativo cambia materialmente la lectura de Day 05;
- si la línea tabular se cierra o si merece una iteración posterior de tuning/resampling.

## CONCLUSIONES

modelo puro:
- propone ranking según patrón aprendido

capa determinista:
- mira ese ranking y dice “en estos casos especiales voy a corregirlo”

problema:
- algunas reglas especiales son buenas en muchos casos, pero no en todos
- entonces corriges 14 veces bien y 3 veces mal
- la media sube, pero rompes la regla de no-harm

En este caso, el modelo ganador es: `V2_TRANSPORT_ONLY_LIGHTGBM_CLASS_WEIGHT_BALANCED_v1`.

Es un modelo puro que mejora las métricas del baseline.